# 01 — Dataset exploration

**Purpose:** inspect physical MAGFiLO images, COCO polygons, annotator variation, intensity, and crop bias.  
**Inputs:** local `data/filament-segmentation-2026/` and optional v1 checkpoint for the final qualitative plot.  
**Outputs:** displayed tables, histograms, and plots; this notebook writes no data or weights.  
**Dependencies:** project `.conda` environment and `src/filament_seg`.  
**Execution:** run cells top to bottom from the repository root or `notebooks/`; the final model plot requires an existing v1 checkpoint.  
**Runtime:** JSON summaries are quick; full-resolution mask sampling and inference can take minutes.  
**Next:** `02_training.ipynb` for checks and explicit training, `04_instance_error_analysis.ipynb` for trained-model errors.

The first annotation set per physical filename is used for binary foreground summaries. Alternative sets remain separate and are compared below. Historical EDA outputs are retained in Git history; current results are recomputed from local data.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'dataset.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'dataset.py').is_file(), f'Project root not found from {cwd}'
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import set_seed, load_config, get_device, get_logger
from filament_seg.dataset import DatasetInspector, SolarFilamentDataset
from filament_seg.preprocessing import ForegroundAwareCrop, get_train_transform, get_val_transform
from filament_seg.model import build_model
from filament_seg.train import load_checkpoint
from filament_seg.inference import predict_tiled, iter_probability_instances

set_seed(42)
logger = get_logger('eda')
print('Environment initialized successfully.')
print('Project Root:', project_root)
print('Python Executable:', sys.executable)

## 1. Dataset Discovery & File Inspection

Scan the dataset directory to verify physical images, annotation files, and file dimensions.

In [ ]:
data_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026'
inspector = DatasetInspector(data_dir)
inspector.print_summary()

train_images_dir = data_dir / 'train' / 'train_images'
test_images_dir = data_dir / 'test' / 'test_images'
train_files = sorted(train_images_dir.glob('*.jpeg'))
test_files = sorted(test_images_dir.glob('*.jpeg'))
print(f'Discovered physical training JPEGs: {len(train_files)}')
print(f'Discovered physical test JPEGs: {len(test_files)}')

## 2. COCO Annotation Integrity & Multi-Annotator Structure

Analyze the ground-truth JSON structure, image-to-annotation mapping, polygon validity, and alternative annotator sets.

In [ ]:
import json
from collections import Counter, defaultdict

annotation_path = data_dir / 'train' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
coco = json.loads(annotation_path.read_text(encoding='utf-8'))
dataset = SolarFilamentDataset(train_images_dir, annotation_path)
records_by_id = {record['id']: record for record in coco['images']}
annotations_by_id = defaultdict(list)
for ann in coco['annotations']:
    annotations_by_id[ann['image_id']].append(ann)

assert len(dataset) == len(train_files) == 707
assert len(coco['images']) == 1154 and len(coco['annotations']) == 8199
assert all(record['height'] == record['width'] == 2048 for record in coco['images'])
assert all(ann['image_id'] in records_by_id for ann in coco['annotations'])
assert len({ann['id'] for ann in coco['annotations']}) == len(coco['annotations'])
print('Physical images, image records, annotations:', len(dataset), len(coco['images']), len(coco['annotations']))
print('Annotation sets per filename:', dict(sorted(Counter(map(len, dataset.images_by_filename.values())).items())))
print('Categories:', coco['categories'])
primary_counts = [len(annotations_by_id[dataset.images_by_filename[name][0]['id']]) for name in dataset.filenames]
print(pd.Series(primary_counts).describe().to_string())


## 3. Foreground Distribution & Class Imbalance Analysis

Evaluate pixel-level filament coverage and contrast across full-resolution solar images.

In [ ]:
sample_limit = min(50, len(dataset))
foreground_fractions = []

for idx in range(sample_limit):
    sample = dataset[idx]
    mask = sample['mask'].squeeze().numpy()
    fg_fraction = (mask > 0).sum() / mask.size
    foreground_fractions.append(fg_fraction)

print(f'Sampled {sample_limit} images:')
print(f'  Min foreground fraction: {min(foreground_fractions)*100:.3f}%')
print(f'  Max foreground fraction: {max(foreground_fractions)*100:.3f}%')
print(f'  Mean foreground fraction: {np.mean(foreground_fractions)*100:.3f}%')
print('Mean sampled foreground coverage is under 0.5%; individual images can exceed that.')

## 4. Preprocessing & Crop Diagnostics (Foreground-Aware vs. Center Crop)

Compare the training foreground-aware crop with center-crop to demonstrate the severe empty-patch deficit identified in the audit.

In [ ]:
# Compare deterministic center crops with sampled foreground-aware crops.
empty_center_crops = 0
sampled_count = min(30, len(dataset))
for idx in range(sampled_count):
    full_mask = dataset[idx]['mask'].squeeze().numpy()
    h, w = full_mask.shape
    center_crop = full_mask[h//2-256:h//2+256, w//2-256:w//2+256]
    empty_center_crops += int(center_crop.sum() == 0)
print(f'Empty center crops: {empty_center_crops}/{sampled_count} ({empty_center_crops/sampled_count:.1%})')
print('Validation uses deterministic crops; full-resolution validation is needed for competition-facing diagnostics.')


## 5. Model Prediction & Failure-Case Visualization

Load the best baseline checkpoint (`best_model.pth`) and visualize predictions across key failure modes (TP, FP, FN, and instance separation).

In [ ]:
device = get_device()
model = build_model(pretrained=False).to(device)
ckpt_path = project_root / 'checkpoints' / 'unet_resnet34' / 'best_model.pth'
load_checkpoint(ckpt_path, model, map_location=device)
model.eval()

# Evaluate sample image at full resolution
sample_idx = 0
sample = dataset[sample_idx]
img = sample['image'].squeeze().numpy()
gt_mask = sample['mask'].squeeze().numpy().astype(np.uint8)

prob_map = predict_tiled(model, sample['image'], tile_size=512, overlap=128, device=device)
bin_pred = (prob_map >= 0.75).astype(np.uint8)
pred_instances = list(iter_probability_instances(prob_map, threshold=0.75, min_area=300))

fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=120)
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Original: {sample["filename"]}')
axes[0].axis('off')

axes[1].imshow(gt_mask, cmap='magma')
axes[1].set_title('Ground Truth Mask')
axes[1].axis('off')

im_p = axes[2].imshow(prob_map, cmap='inferno')
axes[2].set_title('Predicted Probability Map')
axes[2].axis('off')
plt.colorbar(im_p, ax=axes[2], fraction=0.046, pad=0.04)

# Error overlay: Green=TP, Red=FP, Blue=FN
err = np.zeros((gt_mask.shape[0], gt_mask.shape[1], 3), dtype=np.uint8)
err[:] = (img * 150).astype(np.uint8)[..., None]
err[(bin_pred == 1) & (gt_mask == 1)] = [0, 255, 0]
err[(bin_pred == 1) & (gt_mask == 0)] = [255, 0, 0]
err[(bin_pred == 0) & (gt_mask == 1)] = [0, 100, 255]
axes[3].imshow(err)
axes[3].set_title(f'Error Overlay (Pred instances: {len(pred_instances)})')
axes[3].axis('off')
plt.tight_layout()
plt.show()

## Annotation geometry and data quality

**Observation to test:** filament polygons are thin, unevenly sized, and sometimes touch image borders. **Risk:** small fragments can be lost during cropping or area filtering. **Pipeline implication:** retain native-resolution instance masks for evaluation and report the area distribution. The summaries below use COCO metadata; they do not rasterize every annotation.

In [ ]:
areas = np.asarray([a['area'] for a in coco['annotations']], dtype=float)
bboxes = np.asarray([a['bbox'] for a in coco['annotations']], dtype=float)
bad_polygons = [a['id'] for a in coco['annotations'] if not isinstance(a.get('segmentation'), list) or any(len(p) < 6 or len(p) % 2 for p in a['segmentation'])]
border_touching = ((bboxes[:, 0] <= 0) | (bboxes[:, 1] <= 0) | (bboxes[:, 0] + bboxes[:, 2] >= 2048) | (bboxes[:, 1] + bboxes[:, 3] >= 2048)).sum()
print('Invalid polygon structure:', len(bad_polygons), 'Border touching boxes:', int(border_touching))
print('Area quantiles:', np.quantile(areas, [0, .1, .5, .9, 1]))
print('BBox aspect ratio quantiles:', np.quantile(np.maximum(bboxes[:,2], bboxes[:,3]) / np.maximum(1, np.minimum(bboxes[:,2], bboxes[:,3])), [.1,.5,.9]))
fig, axes = plt.subplots(1,2,figsize=(10,3))
axes[0].hist(np.log1p(areas), bins=40); axes[0].set(xlabel='log(1 + COCO area)', ylabel='annotations')
axes[1].hist(primary_counts,bins=25); axes[1].set(xlabel='primary-set instances per image',ylabel='images')
plt.tight_layout(); plt.show()

## Image intensity and split integrity

**Observation to test:** images have varying contrast and severe foreground imbalance. **Risk:** normalization and random-image validation may conceal low-contrast or temporally similar cases. **Pipeline implication:** keep one normalization path and compare a date-grouped holdout in future experiments.

In [ ]:
image_stats = []
for path in train_files[:30]:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    assert image is not None and image.shape == (2048, 2048)
    image_stats.append({'filename':path.name,'mean':float(image.mean()),'std':float(image.std()),'p01':float(np.percentile(image,1)),'p99':float(np.percentile(image,99))})
print(pd.DataFrame(image_stats).describe().to_string())
indices = np.random.default_rng(42).permutation(len(dataset))
train_names = {dataset.filenames[i] for i in indices[:565]}
val_names = {dataset.filenames[i] for i in indices[565:]}
assert len(train_names)==565 and len(val_names)==142 and not train_names & val_names
shared_dates = {n[:8] for n in train_names} & {n[:8] for n in val_names}
print('Train/val physical images:', len(train_names), len(val_names), 'shared dates:', len(shared_dates))

## Alternative annotation sets and representative masks

**Observation to test:** multiple COCO image records can annotate the same JPEG differently. **Risk:** treating them as independent images leaks physical content and overstates sample size. **Pipeline implication:** split by filename, retain alternatives, and report annotator sensitivity alongside primary-set PQ.

In [ ]:
multi_indices = [i for i,name in enumerate(dataset.filenames) if len(dataset.images_by_filename[name]) > 1][:3]
for idx in multi_indices:
    name = dataset.filenames[idx]
    sets = [[inst['mask'] for inst in dataset.iter_instances(idx,k)] for k in range(len(dataset.images_by_filename[name]))]
    unions = [np.logical_or.reduce(masks) if masks else np.zeros((2048,2048),bool) for masks in sets]
    agreement = [float(np.logical_and(unions[0],other).sum()/max(1,np.logical_or(unions[0],other).sum())) for other in unions[1:]]
    print(name, 'instance counts:', list(map(len,sets)), 'foreground IoU vs set 0:', agreement)
    del sets, unions
sample = dataset[multi_indices[0]]
instances = [inst['mask'] for inst in dataset.iter_instances(multi_indices[0],0)]
color = np.zeros((*sample['mask'].shape[-2:],3),dtype=np.uint8)
for k,mask in enumerate(instances): color[mask.astype(bool)] = ((67*k+80)%255,(113*k+120)%255,(191*k+40)%255)
fig,axes=plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(sample['image'].squeeze(),cmap='gray'); axes[0].set_title('Grayscale image')
axes[1].imshow(sample['mask'].squeeze(),cmap='gray'); axes[1].set_title('Binary foreground')
axes[2].imshow(color); axes[2].set_title(f'{len(instances)} separate instances')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## EDA conclusions

The image-level split avoids exact filename overlap, while alternative annotations and shared capture dates remain evaluation considerations. Inspect the printed area, intensity, crop, and agreement distributions before changing thresholds or sampling policy. The optional v1 prediction plot above is historical context, not a v2 result. Full-resolution v2 errors and controlled tuning live in notebooks 04 and 05.